# Export Satellite Imagery for Training Dataset

In [ ]:
import ee
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from eeutils import LandsatSR, Sentinel2SR, SpatialCovariates, get_patch, get_neighbourhood, plot_neighbourhood, get_utm_epsg, extract_patches, extract_spatial_covariates

In [ ]:
# --- PATHS
HOME_DIR = Path.home() / 'Projects/mapping-asia-wealth'                        # project root dir
LABELS_DIR = HOME_DIR / 'data-clean/dhs'                                       # directory with DHS cluster labels
LABELS_PATH = LABELS_DIR / 'dhs_clusters_asia.csv'                             # path to cluster ids and centroids
LANDSAT_IMAGERY_OUTDIR = HOME_DIR / 'data-raw/imagery/training/landsat'        # path to save exported Landsat image patches
SENTINEL_IMAGERY_OUTDIR = HOME_DIR / 'data-raw/imagery/training/sentinel'      # ... for Sentinel 2 imagery
COVARIATES_OUTDIR = HOME_DIR / 'data-raw/covariates/training'                  # ... for spatial covariates

In [ ]:
# --- EXPORT PARAMS
GC_PROJECT = 'historical-infrastructure'                                # Google Cloud project attached to Earth Engine
N_CONCURRENT_REQUESTS = 40                                              # Earth Engine default limit is 40
OVERWRITE_EXISTING_PATCHES = False                                      # skip patch if already downloaded

In [ ]:
ee.Authenticate()
ee.Initialize(project=GC_PROJECT, opt_url='https://earthengine-highvolume.googleapis.com')

In [ ]:
# Load DHS labels
dhs_labels = pd.read_csv(LABELS_PATH)

In [ ]:
# Set the location to test patch extraction
manila = (120.9842, 14.5995)
manila_crs = get_utm_epsg(manila)

In [ ]:
LANDSAT_IMAGERY_OUTDIR.mkdir(exist_ok=True, parents=True)
SENTINEL_IMAGERY_OUTDIR.mkdir(exist_ok=True, parents=True)
COVARIATES_OUTDIR.mkdir(exist_ok=True, parents=True)

In [ ]:
# --- LANDSAT IMAGE PARAMS
LANDSAT_BANDS = ['BLUE', 'GREEN', 'RED', 'NIR', 'SWIR1', 'SWIR2']      # bands to select
LANDSAT_SCALE = 30                                                     # spatial resolution of export in meters: 30m/px
LANDSAT_PATCH_SIZE = 224                                               # patch diameter in pixels
LANDSAT_NODATA_VAL = 0                                                 # if only masked pixels, replace pixel w. zero

# --- SENTINEL 2 PARAMS
SENTINEL2_BANDS = ["B2", "B3", "B4", "B5", "B6", "B7", "B8", "B11", "B12"]      # ...
SENTINEL2_SCALE = 10                                                            # ...
SENTINEL2_PATCH_SIZE = 512                                                      # ...
SENTINEL2_NODATA_VAL = 0                                                        # ...

# --- SPATIAL COVARIATES PARAMS
LANDSAT_FOOTPRINT = LANDSAT_SCALE * LANDSAT_PATCH_SIZE            # set composite scale at 6.72km to match Landsat footprint
SENTINEL2_FOOTPRINT = SENTINEL2_SCALE * SENTINEL2_PATCH_SIZE      # set composite scale at 5.12km to match Sentinel footprint
COVS_NODATA_VAL = -9999                # if only masked pixels, replace pixel w. -999
LANDCOVER_RES = 500               # downsample DynamicWorld to 500m before aggregating
MAX_PIXELS = 1000                 # use at most 1000 pixels of source imagery per pixel in composite
BEST_EFFORT = True                # use higher-aggregation source imagery if max_pixels would be exceeded

# --- ALPHAEARTH PARAMS
ALPHAEARTH_FOOTPRINT = SENTINEL2_FOOTPRINT


## Landsat Imagery
We'll start by exporting 224x224 pixel patches surrounding each of our DHS cluster centroids. To check that the machinery is working as expected, we'll first visualise a set of patches covering Manila drawm from a Landsat median composite for 2025.

We then launch the export for image patches for our DHS clusters in batches. For each DHS Survey Round (i.e. country-year pair) we first create a 3-year Median Composite centred on the survey year [t-1, t+1]. For those DHS surveys for which data collection spanned two years, we centre the composite on the first survey year. We then extract patches from this composite centred on each cluster centroid.

In [ ]:
# Create median-composite for 2024 using all available Landsat imagery
ls_manila = LandsatSR(start_date="2025-01-01", end_date="2025-12-31", bands=LANDSAT_BANDS,
                      rescale_bands=False, mask_clouds=True).images.median().uint16()
landsat_example = get_neighbourhood(image=ls_manila, pt=manila, patch_scale=30, patch_crs=manila_crs)

In [ ]:
# Check True-Colour
plot_neighbourhood(patches=landsat_example, levels=2, bands=["RED", "GREEN", "BLUE"], vis_min=7500, vis_max=16000)

In [ ]:
# Check False-Colour Infrared
plot_neighbourhood(patches=landsat_example, levels=2, bands=["NIR", "SWIR1", "SWIR2"], vis_min=7500, vis_max=16000)

In [ ]:
# Export Landsat patches centered on DHS Cluster centroids for all available surveys in Asia >= DHS4
dhs_landsat = pd.read_csv(LABELS_PATH)
dhs_landsat['image_name'] = None
for (country_code, survey_year), data in dhs_landsat.groupby(['country_code', 'survey_year']):

    # Create three-year (centred on survey year) median composite pooling Landsat 4-9
    print(f"Getting patches for {country_code} -- {survey_year}:")
    landsat_composite = LandsatSR(start_date=f"{survey_year-1}-01-01", end_date=f"{survey_year+1}-12-31", bands=LANDSAT_BANDS, 
                                  rescale_bands=False, mask_clouds=True).images.median().uint16()

    # Extract patches centred on each DHS cluster
    cluster_pts = list(zip(data['lon'], data['lat']))
    cluster_ids = list(data["patch_id"])
    patch_prefix = f"LANDSAT_{country_code.upper()}{survey_year}_"
    extract_patches(image=landsat_composite, pts=cluster_pts, patch_ids=cluster_ids, patch_size=LANDSAT_PATCH_SIZE, patch_scale=LANDSAT_SCALE,
                    patch_prefix=patch_prefix, output_dir=LANDSAT_IMAGERY_OUTDIR, overwrite_patches=False)

    # Save image paths to labels df
    patch_paths = [patch_prefix + cid + ".tif" for cid in cluster_ids]
    dhs_landsat.loc[(dhs_landsat['country_code'] == country_code) & (dhs_landsat['survey_year'] == survey_year), "image_name"] = patch_paths

dhs_landsat.to_csv(LABELS_DIR / 'asia-training-labels-landsat.csv')

## Sentinel Imagery

In [ ]:
# Get median composite for 2024 using all available Sentinel-2 imagery
s2_manila = Sentinel2SR(start_date="2025-01-01", end_date="2025-12-31", keep_qa=False, rescale=False, bands=SENTINEL2_BANDS).images.median().uint16()
sentinel_example = get_neighbourhood(image=s2_manila, pt=manila, patch_scale=SENTINEL2_SCALE, patch_crs=manila_crs, patch_size=SENTINEL2_PATCH_SIZE)

In [ ]:
# Check True-Colour
plot_neighbourhood(patches=sentinel_example, levels=2, bands=["B4", "B3", "B2"], vis_min=0, vis_max=0.3*10000)

In [ ]:
# Check False-Colour Infrared
plot_neighbourhood(patches=sentinel_example, levels=2, bands=["B8", "B11", "B12"], vis_min=0, vis_max=0.3*10000)

In [ ]:
# Export Landsat patches centered on DHS Cluster centroids for all available surveys in Asia >= DHS4
dhs_sentinel = pd.read_csv(LABELS_PATH)
dhs_sentinel['image_name'] = None
for (country_code, survey_year), data in dhs_sentinel.groupby(['country_code', 'survey_year']):

    # Create annual best quality pixel composite from all available images
    print(f"Getting patches for {country_code} -- {survey_year}:")
    composite_year = max(survey_year, 2018)                                       # clamp at 2018; earliest observations from 2017
    s2 = Sentinel2SR(start_date=f"{composite_year}-01-01", end_date=f"{composite_year}-12-31", 
                     keep_qa=False, rescale=False, bands=SENTINEL2_BANDS).images.median().uint16()

    # Extract patches from composite centred on each DHS cluster
    cluster_pts = list(zip(data['lon'], data['lat']))
    cluster_ids = list(data["patch_id"])
    patch_prefix = f"SENTINEL2_"
    extract_patches(image=s2, pts=cluster_pts, patch_ids=cluster_ids, patch_size=SENTINEL2_PATCH_SIZE, patch_scale=SENTINEL2_SCALE,
                    patch_prefix=patch_prefix, output_dir=SENTINEL_IMAGERY_OUTDIR, overwrite_patches=False)

    # Save image paths to labels df
    patch_paths = [patch_prefix + cid + ".tif" for cid in cluster_ids]
    dhs_sentinel.loc[(dhs_sentinel['country_code'] == country_code) & (dhs_sentinel['survey_year'] == survey_year), "image_name"] = patch_paths

dhs_sentinel.to_csv(LABELS_DIR / 'asia-training-labels-sentinel.csv')

## Spatial Covariates

In [ ]:
# Request Patches from the spatial covariates composite at varying scales
spatial_demo = SpatialCovariates(year=2010, landcover_scale=LANDCOVER_RES, nodata_val=COVS_NODATA_VAL, max_pixels=MAX_PIXELS, best_effort=BEST_EFFORT).all_covariates_comp
manila_patch_500m = get_patch(image=spatial_demo, pt=manila, patch_scale=500, patch_crs=manila_crs, patch_size=52, file_format="NUMPY_NDARRAY")

In [ ]:
# Plot VIIRS -- 500m
vis_min, vis_max = 0, 100
band_500m = manila_patch_500m['viirs_mean'].astype(float)
band_500m = (np.clip(band_500m, vis_min, vis_max) - vis_min) / (vis_max - vis_min)
plt.imshow(band_500m, vmin=0, vmax=1, cmap='gray')

In [ ]:
# Plot Land Cover (Dynamic World) -- 500m
band_500m = manila_patch_500m['lulc_prob_built'].astype(float)
plt.imshow(band_500m, vmin=0, vmax=1)

In [ ]:
# Plot Built Surface
vis_min, vis_max = 0, 5000
band_built_surface = manila_patch_500m["built_surface"].astype(float)
band_built_surface = (np.clip(band_built_surface, vis_min, vis_max) - vis_min) / (vis_max - vis_min)
plt.imshow(band_built_surface, vmin=0, vmax=1, cmap='gray')

In [ ]:
# --- Export Spatial Covariates (for areas matching both Sentinel and Landsat footprints)
landsat_df_list = []
sentinel_df_list = []
for survey_year, data in dhs_labels.groupby('survey_year'):

    # Build composite for survey_year
    spatial_cov = SpatialCovariates(year=survey_year, nodata_val=COVS_NODATA_VAL, landcover_scale=LANDCOVER_RES, max_pixels=MAX_PIXELS, best_effort=BEST_EFFORT).all_covariates_comp

    # Extract covariate summary stats for a
    cluster_pts = list(zip(data['lon'], data['lat'], strict=True))
    cluster_ids = list(data["patch_id"])
    landsat_covs = extract_spatial_covariates(image=spatial_cov, pts=cluster_pts, patch_ids=cluster_ids, patch_scale=LANDSAT_FOOTPRINT)
    sentinel_covs = extract_spatial_covariates(image=spatial_cov, pts=cluster_pts, patch_ids=cluster_ids, patch_scale=SENTINEL2_FOOTPRINT)
    landsat_df_list.append(landsat_covs)
    sentinel_df_list.append(sentinel_covs)


In [ ]:
# --- Clean and Save Spatial Covariates
landsat_covs = pd.concat(landsat_df_list, ignore_index=True)
landsat_covs.to_parquet(COVARIATES_OUTDIR / "landsat-covariates.parquet")
landsat_covs.dropna().to_parquet(COVARIATES_OUTDIR / "landsat-covariates-nona.parquet")

sentinel_covs = pd.concat(sentinel_df_list, ignore_index=True)
sentinel_covs.to_parquet(COVARIATES_OUTDIR / "sentinel-covariates.parquet")
sentinel_covs_nona.dropna().to_parquet(COVARIATES_OUTDIR / "sentinel-covariates-nona.parquet")

## AlphaEarth Embeddings

In [ ]:
# Plot AEF Embeddings
aef_demo = ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL").filterDate('2020-01-01', '2020-12-31').mosaic()
aef_patch_500m = get_patch(image=aef_demo, pt=manila, patch_scale=500, patch_crs=manila_crs, patch_size=52, file_format="NUMPY_NDARRAY")

In [ ]:
plot_neighbourhood(patches=aef_patch_500m, levels=0, bands=['A01', 'A16', 'A09'], vis_min=-0.3, vis_max=0.3)

In [ ]:
# --- Export Spatial Covariates (for areas matching both Sentinel and Landsat footprints)
aef_list = []
for survey_year, data in dhs_labels.groupby('survey_year'):

    survey_year = np.clip(survey_year, a_min=2017, a_max=2024)    # AEF starts in 2017; use first year available
    alpha_earth = ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL").filterDate(f'{survey_year}-01-01', f'{survey_year}-12-31').mosaic()

    cluster_pts = list(zip(data['lon'], data['lat']))
    cluster_ids = list(data["patch_id"])
    aef_embeddings = extract_spatial_covariates(image=alpha_earth, pts=cluster_pts, patch_ids=cluster_ids, patch_scale=ALPHA_EARTH_FOOTPRINT)
    aef_list.append(aef_embeddings)


In [ ]:
# Save AEF Embeddings
aef_df = pd.concat(aef_list, ignore_index=True)
aef_df.to_parquet(COVARIATES_OUTDIR / "alphaearth-foundation-embeddings.parquet")